# Mamba + Workspace POC — Kaggle T4

Run cells B and D (the go/no-go pair) on a Kaggle T4 GPU.

**Setup:** Settings → Accelerator → GPU T4 x2, Internet ON

In [ ]:
# Install dependencies (T4 supports CUDA kernels)
!pip install -q mamba-ssm causal-conv1d einops wandb pyyaml
# If mamba-ssm Triton kernels fail on T4, the pure-PyTorch fallback in model.py works too

In [ ]:
# Upload repo as a Kaggle Dataset, or clone from GitHub
# Option 1: Dataset (upload mamba-poc/ as a dataset, then):
# !cp -r /kaggle/input/mamba-poc/* /kaggle/working/

# Option 2: Git clone (if repo is public)
# !git clone https://github.com/youruser/mamba-poc.git /kaggle/working/mamba-poc

import os
os.chdir('/kaggle/working/mamba-poc')

In [ ]:
# Verify GPU and environment
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Torch: {torch.__version__}")

# T4 is Turing — no bf16, use fp16 with gradient scaling
# The training script auto-detects CUDA and uses it

In [ ]:
# Quick data test
!python data.py

In [ ]:
# Param count check
!python model.py

In [ ]:
# Smoke test (quick sanity check)
!python train.py --config configs/cell_d.yaml --smoke-test

## Cell B — Hybrid Baseline (12 Mamba2 + 2 Attention)

This is the real baseline the workspace must beat. ~12 hours on T4.

In [ ]:
# Set wandb API key (store in Kaggle Secrets)
# import kaggle_secrets
# user_secrets = kaggle_secrets.UserSecretsClient()
# wandb_key = user_secrets.get_secret("wandb_api_key")
# os.environ['WANDB_API_KEY'] = wandb_key

# Train Cell B — full run
# Adjust max_steps for T4: ~2B tokens at ~100K tok/s = ~5.5 hrs
!python train.py --config configs/cell_b.yaml

## Cell D — Full Architecture (Hybrid + Workspace + Recurrent Core)

The cell that decides go/no-go. ~12 hours on T4.

In [ ]:
# Train Cell D — full run
!python train.py --config configs/cell_d.yaml

## Analysis (R2, R3, R4)

In [ ]:
# Run all analyses on Cell D
!python probe.py --checkpoint checkpoints/cellD_latest.pt --config configs/cell_d.yaml --all

In [ ]:
# Save checkpoints to Kaggle output (survives session end)
!cp -r checkpoints /kaggle/working/
!cp -r /kaggle/working/mamba-poc /kaggle/working/mamba-poc-output